# Ancillary analysis: barriers and desired features

Analyze 6 barriers against 15 desired features in both conditional directions. This is an association analysis, not a causal analysis.


In [ ]:
#@title 1. Set input and output paths { display-mode: "form" }
INPUT_DATA_PATH = "/content/drive/MyDrive/SOICT-2026/data/soict_2026_survey_responses.csv" #@param {type:"string"}
OUTPUT_DIR_PATH = "/content/drive/MyDrive/SOICT-2026/results/ancillary" #@param {type:"string"}

# Set only the input and output paths for a standard run.

## 2. Set up the environment


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import sys
import warnings
import zipfile

import numpy as np
import pandas as pd
import scipy
from scipy.stats import fisher_exact, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests

SEED = 20260919
FDR_ALPHA = 0.05
HIGH_THRESHOLD = 4
MIN_CATEGORY_N = 10
np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
INPUT_PATH = Path(INPUT_DATA_PATH).expanduser()
OUTPUT_DIR = Path(OUTPUT_DIR_PATH).expanduser()
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH}\n"
        "Check the Google Drive path in the first code cell."
    )
if INPUT_PATH.resolve() == OUTPUT_DIR.resolve():
    raise ValueError("The output path must be a directory distinct from the input file.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=0.95)
print(f"Input file : {INPUT_PATH}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Colab mode : {IN_COLAB}")

## 3. Define barriers and features


In [ ]:
BARRIER_COLS = [f"PTED{i:02d}" for i in range(1, 7)]
FEATURE_COLS = (
    [f"LPSN{i:02d}" for i in range(1, 6)]
    + [f"SFN{i:02d}" for i in range(1, 5)]
    + [f"TECN{i:02d}" for i in range(1, 7)]
)
ANALYSIS_COLS = BARRIER_COLS + FEATURE_COLS

ITEM_LABELS = {item: {'ENP01': 'I need English to communicate with other people.', 'ENP02': 'I need English to access up-to-date information.', 'ENP03': 'I need English for my future career.', 'ENP04': 'I need English to access scholarship opportunities.', 'ENP05': 'I need English to complete coursework in my field of study.', 'ENP06': 'I need English to meet the language requirements of my academic program.', 'PTED01': 'I listen to and understand lectures in my field that are delivered in English.', 'PTED02': 'I read and understand materials in my field that are written in English.', 'PTED03': 'I use English to answer questions in written examinations for courses in my field.', 'PTED04': 'I use English to discuss topics in my field of study.', 'PTED05': 'I use English to answer questions in oral examinations.', 'PTED06': 'I use English to complete writing assignments for courses in my field.', 'MEU01': 'I read materials in my field of study in English.', 'MEU02': 'I listen to learning content in my field of study in English.', 'MEU03': 'I use English to complete written assignments for courses in my field.', 'MEU04': 'I summarize information from English materials for coursework in my field.', 'MEU05': 'I use English to discuss topics in my field of study.', 'REH01': 'I repeat the material I am learning aloud to remember it.', 'REH02': 'I reread learning materials several times.', 'REH03': 'I remember keywords in English materials to recall important ideas.', 'ELA01': 'I connect new material with what I already know.', 'ELA02': 'I summarize the main ideas of the English materials I have studied.', 'ELA03': 'I combine information from several learning sources.', 'ORG01': 'I identify important ideas in the material I am studying.', 'ORG02': 'I make an outline to organize the material I am studying.', 'ORG03': 'I use tables or diagrams to organize the material I am studying.', 'CRT01': 'I evaluate how convincing the English-language information I encounter is.', 'CRT02': 'I examine evidence supporting an explanation or conclusion.', 'CRT03': 'I consider alternative explanations for a claim or conclusion.', 'MSR01': 'I set specific goals before each independent English study session.', 'MSR02': 'I ask myself questions to check my understanding while studying.', 'MSR03': 'I identify what I do not fully understand while studying English independently.', 'MSR04': 'I change my study method when the current method does not work.', 'TSEM01': 'I reserve specific times during the week for independent English study.', 'TSEM02': 'I maintain the independent English study schedule I set.', 'TSEM03': 'I choose a place to study English that limits distractions.', 'VLS01': 'I use new English words in sentences to remember their meanings and usage.', 'VLS02': 'I say or write new English words repeatedly to remember them.', 'VLS03': 'I use flashcards to remember new English words.', 'VLS04': 'I practice pronouncing new English words to remember their pronunciation.', 'VLS05': 'I review English words I have learned at different times.', 'VLS06': 'I try to infer the meaning of unfamiliar English words.', 'DEL01': 'I use digital tools or resources to learn more about English.', 'DEL02': 'I use technology to sustain English study toward my goals.', 'DEL03': 'I use technology to create opportunities to practice English outside class.', 'DEL04': 'I actively search online for English materials to support my studies.', 'DEL05': 'I use learning applications to practice English topics I need to improve.', 'DEL06': 'I select digital tools that fit my independent English study goals.', 'DEL07': 'I ask an AI tool to explain something I do not understand while studying English independently.', 'LPSN01': 'I need the application to clarify the objective of each learning task.', 'LPSN02': 'I need the application to show my learning progress.', 'LPSN03': 'I need the application to show statistics on my learning results.', 'LPSN04': 'I need the application to order tasks from suitable for my current ability to more difficult.', 'LPSN05': 'I need the application to offer tasks that let me learn with others toward a shared goal.', 'SFN01': 'I need the application to suggest how to continue a task when I encounter difficulties.', 'SFN02': 'I need the application to provide examples when I do not understand how to perform a task.', 'SFN03': 'I need the application to indicate which parts of my response do not meet the requirements.', 'SFN04': 'I need the application to explain how to revise an inadequate response after I finish a task.', 'TECN01': 'I need the application to select learning content that matches my learning goals.', 'TECN02': 'I need the application to present content in a way that matches my understanding.', 'TECN03': 'I need the application to suggest the next task after I complete a learning activity.', 'TECN04': 'I need the application to explain why it makes a learning suggestion.', 'TECN05': 'I need control over which learning history information the application may use.', 'TECN06': 'I need the application to suggest review times based on my learning history.', 'PU01': 'I believe the application could help me complete independent English study tasks faster.', 'PU02': 'I believe the application could improve the quality of my independent English study work.', 'PU03': 'I believe the application could help me complete more study activities in the same time.', 'PU04': 'I believe the application could make my independent English study more effective.', 'BI01': 'I intend to use the application for independent English study in the next few months.', 'BI02': 'I expect to use the application for independent English study when it becomes available in the next few months.', 'BI03': 'I plan to use the application to support independent English study in the next few months.'}[item] for item in ANALYSIS_COLS}

assert len(BARRIER_COLS) == 6
assert len(FEATURE_COLS) == 15
assert len(ITEM_LABELS) == 21

## 4. Load and validate the data


In [ ]:
def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            block = stream.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def read_input(path: Path):
    suffix = path.suffix.lower()
    attempts = []
    if suffix in {".xlsx", ".xls"}:
        book = pd.ExcelFile(path)
        candidates = []
        for sheet in book.sheet_names:
            frame = pd.read_excel(path, sheet_name=sheet)
            candidates.append((len(frame), len(frame.columns), sheet, frame))
        _, _, sheet, frame = max(candidates, key=lambda x: (x[0], x[1]))
        return frame, {"file_type": suffix, "sheet": sheet, "encoding": None, "separator": None}
    if suffix not in {".csv", ".txt"}:
        raise ValueError("The input must be CSV, TXT, XLSX, or XLS")
    for encoding in ("utf-8-sig", "utf-8", "cp1258", "cp1252", "latin1"):
        for separator in (";", ",", "\t"):
            try:
                frame = pd.read_csv(path, encoding=encoding, sep=separator)
                if len(frame.columns) > 1:
                    return frame, {
                        "file_type": suffix,
                        "sheet": None,
                        "encoding": encoding,
                        "separator": repr(separator),
                    }
            except Exception as exc:
                attempts.append(f"{encoding}/{repr(separator)}: {type(exc).__name__}")
    raise ValueError("Unable to read CSV. Attempts: " + " | ".join(attempts[:12]))


def extract_item_code(column_name: str):
    text = str(column_name).strip()
    if text in ANALYSIS_COLS:
        return text
    match = re.search(r"\b(PTED0[1-6]|LPSN0[1-5]|SFN0[1-4]|TECN0[1-6])\b", text)
    return match.group(1) if match else None


def likert_to_numeric(series: pd.Series) -> pd.Series:
    direct = pd.to_numeric(series, errors="coerce")
    if direct.notna().sum() == series.notna().sum():
        return direct
    extracted = series.astype("string").str.extract(r"^\s*([1-5])(?:\s|$|[-:])", expand=False)
    return pd.to_numeric(extracted, errors="coerce").fillna(direct)


raw, source_meta = read_input(INPUT_PATH)
rename_map = {}
for column in raw.columns:
    code_name = extract_item_code(column)
    if code_name:
        if code_name in rename_map.values():
            raise ValueError(f"Multiple columns found for item code {code_name}.")
        rename_map[column] = code_name
    text = str(column).lower()
    if "gender" in text or text.strip() == "gender":
        rename_map[column] = "GENDER"
    if "field of study" in text or text.strip() == "field_of_study":
        rename_map[column] = "FIELD_OF_STUDY"
    if text.strip() == "respondent_id":
        rename_map[column] = "RESPONDENT_ID"

standardized_from_raw_form = not set(ANALYSIS_COLS).issubset(raw.columns)
data = raw.rename(columns=rename_map).copy()
missing_required = sorted(set(ANALYSIS_COLS) - set(data.columns))
if missing_required:
    raise ValueError(
        "Required analysis columns are missing: " + ", ".join(missing_required) +
        ". Supply the cleaned study dataset with the documented schema."
    )

if "RESPONDENT_ID" not in data.columns:
    data.insert(0, "RESPONDENT_ID", [f"ROW_{i+1:04d}" for i in range(len(data))])
for column in ANALYSIS_COLS:
    data[column] = likert_to_numeric(data[column])

invalid_counts = {
    column: int((data[column].notna() & ~data[column].isin([1, 2, 3, 4, 5])).sum())
    for column in ANALYSIS_COLS
}
if sum(invalid_counts.values()):
    raise ValueError(f"Likert values outside 1 to 5: {invalid_counts}")
if data["RESPONDENT_ID"].duplicated().any():
    raise ValueError("RESPONDENT_ID must be unique.")

selected_cols = ["RESPONDENT_ID"]
for optional in ["GENDER", "FIELD_OF_STUDY"]:
    if optional in data.columns:
        selected_cols.append(optional)
selected_cols += ANALYSIS_COLS
df = data[selected_cols].copy()

quality_rows = [
    ("input_rows", len(raw)),
    ("analysis_columns", len(ANALYSIS_COLS)),
    ("barrier_columns", len(BARRIER_COLS)),
    ("feature_columns", len(FEATURE_COLS)),
    ("duplicate_ids", int(df["RESPONDENT_ID"].duplicated().sum())),
    ("invalid_likert_values", int(sum(invalid_counts.values()))),
    ("rows_complete_all_21_items", int(df[ANALYSIS_COLS].notna().all(axis=1).sum())),
    ("standardized_from_raw_form", standardized_from_raw_form),
]
for column in ANALYSIS_COLS:
    quality_rows.append((f"missing_{column}", int(df[column].isna().sum())))
quality = pd.DataFrame(quality_rows, columns=["metric", "value"])
quality.to_csv(OUTPUT_DIR / "01_data_quality.csv", index=False, encoding="utf-8-sig")

print(quality.head(8).to_string(index=False))
if standardized_from_raw_form:
    warnings.warn(
        "Raw form fields were detected; verify the input cleaning provenance before interpreting results."
    )

## 5. Describe the items


In [ ]:
def describe_items(frame: pd.DataFrame, columns: list[str], role: str) -> pd.DataFrame:
    records = []
    for column in columns:
        values = frame[column].dropna()
        records.append({
            "item_code": column,
            "item_text": ITEM_LABELS[column],
            "role": role,
            "n": int(values.size),
            "missing": int(frame[column].isna().sum()),
            "mean": values.mean(),
            "sd": values.std(ddof=1),
            "median": values.median(),
            "q1": values.quantile(0.25),
            "q3": values.quantile(0.75),
            "proportion_ge4": (values >= HIGH_THRESHOLD).mean(),
            "proportion_5": (values == 5).mean(),
        })
    return pd.DataFrame(records)


barrier_summary = describe_items(df, BARRIER_COLS, "Barrier").sort_values(
    ["proportion_ge4", "mean"], ascending=False
)
feature_summary = describe_items(df, FEATURE_COLS, "Feature").sort_values(
    ["proportion_ge4", "mean"], ascending=False
)
barrier_summary.to_csv(OUTPUT_DIR / "02_barrier_summary.csv", index=False, encoding="utf-8-sig")
feature_summary.to_csv(OUTPUT_DIR / "03_feature_summary.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
bplot = barrier_summary.sort_values("proportion_ge4")
fplot = feature_summary.sort_values("proportion_ge4")
axes[0].barh(bplot["item_code"], bplot["proportion_ge4"], color="#C44E52")
axes[0].set(title="High barrier prevalence (4 to 5)", xlabel="Proportion", ylabel="Barrier")
axes[0].set_xlim(0, 1)
axes[1].barh(fplot["item_code"], fplot["proportion_ge4"], color="#4C72B0")
axes[1].set(title="High feature need prevalence (4 to 5)", xlabel="Proportion", ylabel="Feature")
axes[1].set_xlim(0, 1)
fig.suptitle("Barriers and desired features", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "figure_01_descriptive_summary.png", dpi=200, bbox_inches="tight")
plt.show()

display(barrier_summary.round(4))
display(feature_summary.round(4))

## 6. Prepare covariates and ordinal models


In [ ]:
def make_controls(frame: pd.DataFrame) -> pd.DataFrame:
    controls = pd.DataFrame(index=frame.index)
    if "GENDER" in frame.columns:
        gender = frame["GENDER"].fillna("MISSING").astype(str).str.strip()
        controls = pd.concat(
            [controls, pd.get_dummies(gender, prefix="gender", drop_first=True, dtype=float)],
            axis=1,
        )
    if "FIELD_OF_STUDY" in frame.columns:
        field = frame["FIELD_OF_STUDY"].fillna("MISSING").astype(str).str.strip()
        counts = field.value_counts()
        field = field.where(field.map(counts) >= MIN_CATEGORY_N, "OTHER_SMALL")
        controls = pd.concat(
            [controls, pd.get_dummies(field, prefix="field", drop_first=True, dtype=float)],
            axis=1,
        )
    # Remove any constant covariate columns.
    return controls.loc[:, controls.nunique(dropna=False) > 1]


CONTROL_MATRIX = make_controls(df)
print(f"Control dummy variables: {CONTROL_MATRIX.shape[1]}")
print(CONTROL_MATRIX.columns.tolist())


def fit_ordered_pair(frame: pd.DataFrame, outcome: str, predictor: str) -> dict:
    valid = frame[[outcome, predictor]].notna().all(axis=1)
    y = frame.loc[valid, outcome].astype(int)
    x_main = frame.loc[valid, predictor].astype(float).rename(predictor)
    exog = pd.concat([x_main, CONTROL_MATRIX.loc[valid]], axis=1).astype(float)
    exog = exog.loc[:, exog.nunique(dropna=False) > 1]
    base = {
        "outcome": outcome,
        "predictor": predictor,
        "n": int(valid.sum()),
        "n_outcome_levels": int(y.nunique()),
        "n_controls": int(max(0, exog.shape[1] - 1)),
        "converged": False,
        "coefficient": np.nan,
        "se": np.nan,
        "odds_ratio": np.nan,
        "ci95_low": np.nan,
        "ci95_high": np.nan,
        "p_value": np.nan,
        "aic": np.nan,
        "error": "",
    }
    if len(y) < 50 or y.nunique() < 3 or predictor not in exog.columns:
        base["error"] = "Insufficient observations or response variation for the ordinal model"
        return base
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = OrderedModel(y, exog, distr="logit")
            result = model.fit(method="bfgs", maxiter=300, disp=False)
        beta = float(result.params[predictor])
        se = float(result.bse[predictor])
        converged = bool(result.mle_retvals.get("converged", True))
        base.update({
            "converged": converged,
            "coefficient": beta,
            "se": se,
            "odds_ratio": np.exp(beta),
            "ci95_low": np.exp(beta - 1.96 * se),
            "ci95_high": np.exp(beta + 1.96 * se),
            "p_value": float(result.pvalues[predictor]),
            "aic": float(result.aic),
            "error": "" if converged else "Optimizer did not converge",
        })
    except Exception as exc:
        base["error"] = f"{type(exc).__name__}: {str(exc)[:180]}"
    return base


def add_fdr(table: pd.DataFrame, p_col: str = "p_value") -> pd.DataFrame:
    result = table.copy()
    result["fdr_p"] = np.nan
    valid = result[p_col].notna()
    if valid.any():
        result.loc[valid, "fdr_p"] = multipletests(
            result.loc[valid, p_col], alpha=FDR_ALPHA, method="fdr_bh"
        )[1]
    result["fdr_significant"] = result["fdr_p"] < FDR_ALPHA
    return result

## 7. Forward models: feature given barrier


In [ ]:
pair_records = []
forward_records = []
total_pairs = len(BARRIER_COLS) * len(FEATURE_COLS)
counter = 0
for barrier in BARRIER_COLS:
    for feature in FEATURE_COLS:
        counter += 1
        valid = df[[barrier, feature]].dropna()
        rho, rho_p = spearmanr(valid[barrier], valid[feature]) if len(valid) >= 3 else (np.nan, np.nan)
        barrier_high = valid[barrier] >= HIGH_THRESHOLD
        feature_high = valid[feature] >= HIGH_THRESHOLD
        a = int((barrier_high & feature_high).sum())
        b = int((~barrier_high & feature_high).sum())
        c = int((barrier_high & ~feature_high).sum())
        d = int((~barrier_high & ~feature_high).sum())
        # Apply Haldane-Anscombe correction only to descriptive RR/OR when a cell is zero.
        aa, bb, cc, dd = (a + 0.5, b + 0.5, c + 0.5, d + 0.5)
        p_b_given_f = a / (a + b) if (a + b) else np.nan
        p_b_given_not_f = c / (c + d) if (c + d) else np.nan
        risk_ratio = (aa / (aa + bb)) / (cc / (cc + dd))
        conditional_or = (aa * dd) / (bb * cc)
        fisher_p = fisher_exact([[a, b], [c, d]], alternative="two-sided").pvalue
        pair_records.append({
            "barrier_code": barrier,
            "feature_code": feature,
            "n_pair": len(valid),
            "spearman_rho": rho,
            "spearman_p": rho_p,
            "both_high_n": a,
            "feature_high_barrier_low_n": b,
            "barrier_high_feature_low_n": c,
            "both_low_n": d,
            "p_barrier_high_given_feature_high": p_b_given_f,
            "p_barrier_high_given_feature_not_high": p_b_given_not_f,
            "risk_difference_reverse": p_b_given_f - p_b_given_not_f,
            "risk_ratio_reverse": risk_ratio,
            "conditional_odds_ratio": conditional_or,
            "fisher_p": fisher_p,
        })
        forward_records.append(fit_ordered_pair(df, outcome=feature, predictor=barrier))
        if counter % 15 == 0:
            print(f"Forward models completed: {counter}/{total_pairs} models")

pair_stats = pd.DataFrame(pair_records)
for p_col, out_col in [("spearman_p", "spearman_fdr"), ("fisher_p", "fisher_fdr")]:
    pair_stats[out_col] = multipletests(pair_stats[p_col], method="fdr_bh")[1]
pair_stats["barrier_text"] = pair_stats["barrier_code"].map(ITEM_LABELS)
pair_stats["feature_text"] = pair_stats["feature_code"].map(ITEM_LABELS)
pair_stats.to_csv(OUTPUT_DIR / "04_pair_descriptives.csv", index=False, encoding="utf-8-sig")

forward = add_fdr(pd.DataFrame(forward_records))
forward = forward.rename(columns={
    "outcome": "feature_code",
    "predictor": "barrier_code",
    "coefficient": "forward_coefficient",
    "se": "forward_se",
    "odds_ratio": "forward_odds_ratio",
    "ci95_low": "forward_ci95_low",
    "ci95_high": "forward_ci95_high",
    "p_value": "forward_p",
    "fdr_p": "forward_fdr",
    "fdr_significant": "forward_fdr_significant",
    "converged": "forward_converged",
    "error": "forward_error",
})
forward["barrier_text"] = forward["barrier_code"].map(ITEM_LABELS)
forward["feature_text"] = forward["feature_code"].map(ITEM_LABELS)
forward.to_csv(OUTPUT_DIR / "05_forward_barrier_to_feature.csv", index=False, encoding="utf-8-sig")
display(forward.sort_values("forward_fdr").head(20).round(4))

## 8. Reverse models: barrier given feature


In [ ]:
reverse_records = []
counter = 0
for feature in FEATURE_COLS:
    for barrier in BARRIER_COLS:
        counter += 1
        reverse_records.append(fit_ordered_pair(df, outcome=barrier, predictor=feature))
        if counter % 18 == 0:
            print(f"Reverse models completed: {counter}/{total_pairs} models")

reverse = add_fdr(pd.DataFrame(reverse_records))
reverse = reverse.rename(columns={
    "outcome": "barrier_code",
    "predictor": "feature_code",
    "coefficient": "reverse_coefficient",
    "se": "reverse_se",
    "odds_ratio": "reverse_odds_ratio",
    "ci95_low": "reverse_ci95_low",
    "ci95_high": "reverse_ci95_high",
    "p_value": "reverse_p",
    "fdr_p": "reverse_fdr",
    "fdr_significant": "reverse_fdr_significant",
    "converged": "reverse_converged",
    "error": "reverse_error",
})
reverse["barrier_text"] = reverse["barrier_code"].map(ITEM_LABELS)
reverse["feature_text"] = reverse["feature_code"].map(ITEM_LABELS)
reverse.to_csv(OUTPUT_DIR / "06_reverse_feature_to_barrier.csv", index=False, encoding="utf-8-sig")
display(reverse.sort_values("reverse_fdr").head(20).round(4))

## 9. Combine both directions


In [ ]:
synthesis = (
    pair_stats.merge(
        forward[[
            "barrier_code", "feature_code", "forward_coefficient", "forward_odds_ratio",
            "forward_ci95_low", "forward_ci95_high", "forward_p", "forward_fdr",
            "forward_fdr_significant", "forward_converged", "forward_error"
        ]],
        on=["barrier_code", "feature_code"], how="left"
    ).merge(
        reverse[[
            "barrier_code", "feature_code", "reverse_coefficient", "reverse_odds_ratio",
            "reverse_ci95_low", "reverse_ci95_high", "reverse_p", "reverse_fdr",
            "reverse_fdr_significant", "reverse_converged", "reverse_error"
        ]],
        on=["barrier_code", "feature_code"], how="left"
    )
)


def classify_evidence(row):
    values = [row["spearman_rho"], row["forward_coefficient"], row["reverse_coefficient"]]
    if any(pd.isna(value) for value in values):
        return "C: insufficient evidence"
    f_sig = bool(row["forward_fdr_significant"])
    r_sig = bool(row["reverse_fdr_significant"])
    all_positive = all(value > 0 for value in values)
    sign_conflict = (row["forward_coefficient"] * row["reverse_coefficient"] < 0) or (
        row["spearman_rho"] * row["forward_coefficient"] < 0
    )
    significant_negative = (
        (f_sig and row["forward_coefficient"] < 0)
        or (r_sig and row["reverse_coefficient"] < 0)
    )
    if sign_conflict or significant_negative:
        return "D: discordant or negative association"
    if all_positive and f_sig and r_sig:
        return "A: both conditional models supported"
    if all_positive and (f_sig or r_sig):
        return "B: partial evidence"
    return "C: insufficient evidence"


synthesis["evidence_class"] = synthesis.apply(classify_evidence, axis=1)
synthesis["evidence_rank"] = synthesis["evidence_class"].str[0].map({"A": 1, "B": 2, "C": 3, "D": 4})
synthesis["mean_log_odds"] = (
    synthesis["forward_coefficient"] + synthesis["reverse_coefficient"]
) / 2
synthesis = synthesis.sort_values(
    ["evidence_rank", "forward_fdr", "reverse_fdr", "spearman_rho"],
    ascending=[True, True, True, False],
)
synthesis.to_csv(OUTPUT_DIR / "07_cross_synthesis_90_pairs.csv", index=False, encoding="utf-8-sig")

barrier_to_feature = synthesis.sort_values(
    ["barrier_code", "evidence_rank", "forward_fdr", "spearman_rho"],
    ascending=[True, True, True, False],
).copy()
barrier_to_feature["rank_within_barrier"] = barrier_to_feature.groupby("barrier_code").cumcount() + 1
barrier_to_feature.to_csv(
    OUTPUT_DIR / "08_barrier_to_feature_ranking.csv", index=False, encoding="utf-8-sig"
)

feature_to_barrier = synthesis.sort_values(
    ["feature_code", "evidence_rank", "reverse_fdr", "spearman_rho"],
    ascending=[True, True, True, False],
).copy()
feature_to_barrier["rank_within_feature"] = feature_to_barrier.groupby("feature_code").cumcount() + 1
feature_to_barrier.to_csv(
    OUTPUT_DIR / "09_feature_to_barrier_ranking.csv", index=False, encoding="utf-8-sig"
)

print(synthesis["evidence_class"].value_counts().to_string())
display(synthesis[[
    "barrier_code", "feature_code", "spearman_rho", "forward_odds_ratio",
    "forward_fdr", "reverse_odds_ratio", "reverse_fdr",
    "p_barrier_high_given_feature_high", "risk_difference_reverse", "evidence_class"
]].head(30).round(4))

## 10. Plot the 6 by 15 matrix


In [ ]:
rho_matrix = synthesis.pivot(index="barrier_code", columns="feature_code", values="spearman_rho")
rho_matrix = rho_matrix.reindex(index=BARRIER_COLS, columns=FEATURE_COLS)
plt.figure(figsize=(16, 6))
sns.heatmap(rho_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-0.35, vmax=0.35)
plt.title("Barrier and feature association matrix (Spearman)", fontweight="bold")
plt.xlabel("Desired feature")
plt.ylabel("Barrier")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_02_spearman_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

class_map = {
    "A: both conditional models supported": 4,
    "B: partial evidence": 3,
    "C: insufficient evidence": 2,
    "D: discordant or negative association": 1,
}
class_matrix = synthesis.assign(class_score=synthesis["evidence_class"].map(class_map)).pivot(
    index="barrier_code", columns="feature_code", values="class_score"
).reindex(index=BARRIER_COLS, columns=FEATURE_COLS)
annotation = synthesis.assign(short=synthesis["evidence_class"].str[0]).pivot(
    index="barrier_code", columns="feature_code", values="short"
).reindex(index=BARRIER_COLS, columns=FEATURE_COLS)
plt.figure(figsize=(16, 6))
sns.heatmap(
    class_matrix, annot=annotation, fmt="", cmap=["#C44E52", "#D9D9D9", "#DDCC77", "#55A868"],
    vmin=1, vmax=4, cbar_kws={"ticks": [1.4, 2.1, 2.9, 3.6], "label": "D  to  evidence class  to  A"},
    linewidths=0.5, linecolor="white",
)
plt.title("Cross-direction synthesis for 90 pairs", fontweight="bold")
plt.xlabel("Desired feature")
plt.ylabel("Barrier")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_03_cross_evidence_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

## 11. Export results and integrity manifest


In [ ]:
workbook_path = OUTPUT_DIR / "10_barrier_feature_results.xlsx"
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    quality.to_excel(writer, sheet_name="data_quality", index=False)
    barrier_summary.to_excel(writer, sheet_name="barrier_summary", index=False)
    feature_summary.to_excel(writer, sheet_name="feature_summary", index=False)
    pair_stats.to_excel(writer, sheet_name="pair_descriptives", index=False)
    forward.to_excel(writer, sheet_name="forward_models", index=False)
    reverse.to_excel(writer, sheet_name="reverse_models", index=False)
    synthesis.to_excel(writer, sheet_name="cross_synthesis", index=False)
    barrier_to_feature.to_excel(writer, sheet_name="barrier_to_feature", index=False)
    feature_to_barrier.to_excel(writer, sheet_name="feature_to_barrier", index=False)

class_counts = synthesis["evidence_class"].value_counts().to_dict()
confirmed = synthesis[synthesis["evidence_class"] == "A: both conditional models supported"]
partial = synthesis[synthesis["evidence_class"] == "B: partial evidence"]
conflicted = synthesis[synthesis["evidence_class"] == "D: discordant or negative association"]

report_lines = [
    "# Ancillary report: barriers and features",
    "",
    f"- Input file: `{INPUT_PATH.name}`",
    f"- Input rows: **{len(raw)}**",
    f"- Pairs analyzed: **{len(synthesis)}**",
    f"- Converged forward models: **{int(forward['forward_converged'].sum())}/{len(forward)}**",
    f"- Converged reverse models: **{int(reverse['reverse_converged'].sum())}/{len(reverse)}**",
    "",
    "## Evidence classes",
    "",
]
for label in [
    "A: both conditional models supported", "B: partial evidence",
    "C: insufficient evidence", "D: discordant or negative association"
]:
    report_lines.append(f"- {label}: **{class_counts.get(label, 0)} pairs**")

report_lines += ["", "## Class A pairs", ""]
if confirmed.empty:
    report_lines.append("No pair met all class A criteria.")
else:
    for _, row in confirmed.iterrows():
        report_lines.append(
            f"- **{row['barrier_code']} → {row['feature_code']}**: "
            f"rho={row['spearman_rho']:.3f}; forward OR={row['forward_odds_ratio']:.3f} "
            f"(FDR={row['forward_fdr']:.4g}); reverse OR={row['reverse_odds_ratio']:.3f} "
            f"(FDR={row['reverse_fdr']:.4g})."
        )
report_lines += ["", "## Interpretation limits", ""]
report_lines += [
    "1. Forward and reverse directions condition the same cross sectional observations in different ways; they are not independent causal evidence.",
    "2. Statistical significance does not imply a large effect or practical importance.",
    "3. Interpret class D pairs after inspecting signs, distributions, and stability.",
    "4. Results on a raw form export cannot replace the study cleaning and sample definition.",
    "5. This notebook does not analyze technical architecture.",
]
(OUTPUT_DIR / "analysis_summary.md").write_text("\n".join(report_lines), encoding="utf-8")

manifest = {
    "analysis": "Ancillary bidirectional barrier-feature analysis",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_file": str(INPUT_PATH),
    "input_sha256": sha256_file(INPUT_PATH),
    "output_directory": str(OUTPUT_DIR),
    "source_metadata": source_meta,
    "standardized_from_raw_form": standardized_from_raw_form,
    "seed": SEED,
    "fdr_alpha": FDR_ALPHA,
    "high_threshold": HIGH_THRESHOLD,
    "min_category_n": MIN_CATEGORY_N,
    "n_rows": len(df),
    "n_barriers": len(BARRIER_COLS),
    "n_features": len(FEATURE_COLS),
    "n_pairs": len(synthesis),
    "controls_used": CONTROL_MATRIX.columns.tolist(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

zip_path = OUTPUT_DIR / "barrier_feature_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file() and path != zip_path:
            archive.write(path, arcname=path.name)

print("\nANCILLARY ANALYSIS COMPLETE")
print(f"Workbook : {workbook_path}")
print(f"Report : {OUTPUT_DIR / 'analysis_summary.md'}")
print(f"ZIP : {zip_path}")
print("\nPairs by evidence class:")
print(pd.Series(class_counts).to_string())

## 12. Read the results

The forward and reverse models describe different conditionings of the same observational data.
